# Author Embeddings Upload to Vector Database

This notebook loads author and works data from Azure Blob Storage and creates author embeddings based on their paper abstracts. The embeddings are uploaded to a local Milvus vector database using the `/authors/embeddings` endpoint.

## Overview
1. Connect to Azure Blob Storage
2. Load author and works data
3. Group works by author to collect abstracts
4. Upload author embeddings to the vector database

## 1. Install and Import Dependencies

In [1]:
# Import required libraries
import os
import json
import requests
from collections import defaultdict
from typing import Dict, List, Optional
from azure.storage.blob import BlobServiceClient
from tqdm.auto import tqdm
import time

c:\Users\Ethan Shafer\AppData\Local\pypoetry\Cache\virtualenvs\dtic-Cv-TFI6J-py3.13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configure Azure Storage and Vector DB Connection

In [2]:
# Azure Storage Configuration
AZURE_STORAGE_CONNECTION_STRING = os.environ.get('AZURE_STORAGE_CONNECTION_STRING')
CONTAINER_NAME = 'clean'  # Container with cleaned data
AUTHORS_PREFIX = 'dtic/authors/'
WORKS_PREFIX = 'dtic/works/'

# Vector DB Configuration
VECTOR_DB_BASE_URL = 'http://localhost:8002'
EMBEDDINGS_ENDPOINT = f'{VECTOR_DB_BASE_URL}/authors/embeddings'

# Verify connection string
if not AZURE_STORAGE_CONNECTION_STRING:
    raise ValueError("AZURE_STORAGE_CONNECTION_STRING environment variable not set!")

print("✓ Configuration loaded")
print(f"  Container: {CONTAINER_NAME}")
print(f"  Authors prefix: {AUTHORS_PREFIX}")
print(f"  Works prefix: {WORKS_PREFIX}")
print(f"  Vector DB: {VECTOR_DB_BASE_URL}")

✓ Configuration loaded
  Container: clean
  Authors prefix: dtic/authors/
  Works prefix: dtic/works/
  Vector DB: http://localhost:8002


## 3. Connect to Azure Blob Storage

In [3]:
# Initialize Blob Service Client
blob_service_client = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

print("✓ Connected to Azure Blob Storage")

✓ Connected to Azure Blob Storage


## 4. Load Authors from Blob Storage

In [4]:
def load_authors_from_blob(container_client, prefix: str, max_authors: Optional[int] = None) -> Dict[str, dict]:
    """
    Load author data from Azure Blob Storage.
    
    Args:
        container_client: Azure container client
        prefix: Blob prefix for authors (e.g., 'dtic/authors/')
        max_authors: Maximum number of authors to load (None = all)
        
    Returns:
        Dictionary mapping author_id to author data
    """
    authors = {}
    blob_list = container_client.list_blobs(name_starts_with=prefix)
    
    print(f"Loading authors from {prefix}...")
    
    for i, blob in enumerate(tqdm(blob_list)):
        if max_authors and i >= max_authors:
            break
            
        # Download blob content
        blob_client = container_client.get_blob_client(blob.name)
        blob_data = blob_client.download_blob().readall()
        
        try:
            author_data = json.loads(blob_data)
            author_id = author_data.get('id')
            if author_id:
                authors[author_id] = author_data
        except json.JSONDecodeError as e:
            print(f"Error decoding {blob.name}: {e}")
            continue
    
    print(f"✓ Loaded {len(authors)} authors")
    return authors

# Load authors (set max_authors to limit for testing, None for all)
authors = load_authors_from_blob(container_client, AUTHORS_PREFIX, max_authors=None)
print(f"\nTotal authors loaded: {len(authors)}")

Loading authors from dtic/authors/...


3719it [02:13, 27.92it/s]


KeyboardInterrupt: 

## 5. Load Works from Blob Storage

In [ ]:
def load_works_from_blob(container_client, prefix: str, max_works: Optional[int] = None) -> List[dict]:
    """
    Load works data from Azure Blob Storage.
    
    Args:
        container_client: Azure container client
        prefix: Blob prefix for works (e.g., 'dtic/works/')
        max_works: Maximum number of works to load (None = all)
        
    Returns:
        List of work dictionaries
    """
    works = []
    blob_list = container_client.list_blobs(name_starts_with=prefix)
    
    print(f"Loading works from {prefix}...")
    
    for i, blob in enumerate(tqdm(blob_list)):
        if max_works and i >= max_works:
            break
            
        # Download blob content
        blob_client = container_client.get_blob_client(blob.name)
        blob_data = blob_client.download_blob().readall()
        
        try:
            work_data = json.loads(blob_data)
            if work_data.get('abstract'):  # Only include works with abstracts
                works.append(work_data)
        except json.JSONDecodeError as e:
            print(f"Error decoding {blob.name}: {e}")
            continue
    
    print(f"✓ Loaded {len(works)} works with abstracts")
    return works

# Load works (set max_works to limit for testing, None for all)
works = load_works_from_blob(container_client, WORKS_PREFIX, max_works=None)
print(f"\nTotal works with abstracts: {len(works)}")

Loading works from dtic/works/...


16007it [05:48, 48.93it/s]

## 6. Group Works by Author

In [ ]:
def group_abstracts_by_author(works: List[dict]) -> Dict[str, List[str]]:
    """
    Group abstracts by author ID.
    
    Args:
        works: List of work dictionaries
        
    Returns:
        Dictionary mapping author_id to list of abstracts
    """
    author_abstracts = defaultdict(list)
    
    print("Grouping abstracts by author...")
    for work in tqdm(works):
        abstract = work.get('abstract', '').strip()
        if not abstract:
            continue
            
        # Each work can have multiple authors
        authors_list = work.get('authors', [])
        for author_entry in authors_list:
            author_id = author_entry.get('author_id')
            if author_id:
                author_abstracts[author_id].append(abstract)
    
    print(f"✓ Grouped abstracts for {len(author_abstracts)} authors")
    return dict(author_abstracts)

# Group abstracts
author_abstracts = group_abstracts_by_author(works)

# Show statistics
abstracts_per_author = [len(abstracts) for abstracts in author_abstracts.values()]
print(f"\nStatistics:")
print(f"  Authors with abstracts: {len(author_abstracts)}")
print(f"  Total abstract-author pairs: {sum(abstracts_per_author)}")
print(f"  Average abstracts per author: {sum(abstracts_per_author) / len(abstracts_per_author):.2f}")
print(f"  Max abstracts for one author: {max(abstracts_per_author)}")
print(f"  Min abstracts for one author: {min(abstracts_per_author)}")

## 7. Test Vector DB Connection

In [ ]:
# Test connection to vector DB
try:
    health_response = requests.get(f"{VECTOR_DB_BASE_URL}/health", timeout=5)
    health_response.raise_for_status()
    health_data = health_response.json()
    
    print("✓ Vector DB is healthy")
    print(f"  Status: {health_data.get('status')}")
    print(f"  Milvus connected: {health_data.get('milvus_connected')}")
    print(f"  Collections: {health_data.get('collections', [])}")
except requests.exceptions.RequestException as e:
    print(f"✗ Failed to connect to Vector DB: {e}")
    print(f"  Make sure the service is running at {VECTOR_DB_BASE_URL}")
    raise

## 8. Upload Author Embeddings to Vector DB

In [ ]:
def upload_author_embedding(author_id: str, author_name: str, abstracts: List[str], 
                           endpoint: str, timeout: int = 30) -> dict:
    """
    Upload author embedding to vector database.
    
    Args:
        author_id: Author's unique ID
        author_name: Author's name
        abstracts: List of paper abstracts
        endpoint: API endpoint URL
        timeout: Request timeout in seconds
        
    Returns:
        Response dictionary from the API
    """
    payload = {
        "author_id": author_id,
        "author_name": author_name,
        "abstracts": abstracts
    }
    
    response = requests.post(endpoint, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()

# Upload embeddings for all authors
results = {
    'success': [],
    'failed': [],
    'total': 0
}

print(f"Uploading embeddings for {len(author_abstracts)} authors...")
print("This may take a while depending on the number of authors and abstracts...\n")

for author_id, abstracts in tqdm(author_abstracts.items(), desc="Uploading embeddings"):
    results['total'] += 1
    
    # Get author name from authors dict
    author_data = authors.get(author_id, {})
    author_name = author_data.get('name', 'Unknown Author')
    
    try:
        response = upload_author_embedding(
            author_id=author_id,
            author_name=author_name,
            abstracts=abstracts,
            endpoint=EMBEDDINGS_ENDPOINT
        )
        
        results['success'].append({
            'author_id': author_id,
            'author_name': author_name,
            'num_abstracts': len(abstracts),
            'response': response
        })
        
    except requests.exceptions.RequestException as e:
        results['failed'].append({
            'author_id': author_id,
            'author_name': author_name,
            'error': str(e)
        })
        print(f"\n✗ Failed to upload {author_id} ({author_name}): {e}")
    
    # Small delay to avoid overwhelming the server
    time.sleep(0.1)

print(f"\n{'='*60}")
print(f"Upload Complete!")
print(f"{'='*60}")
print(f"Total authors processed: {results['total']}")
print(f"✓ Successful uploads: {len(results['success'])}")
print(f"✗ Failed uploads: {len(results['failed'])}")

if results['failed']:
    print(f"\nFailed uploads:")
    for failure in results['failed'][:10]:  # Show first 10 failures
        print(f"  - {failure['author_name']} ({failure['author_id']}): {failure['error']}")

## 9. Verify Upload - Sample Some Authors

In [ ]:
# Show details of successfully uploaded authors
if results['success']:
    print("Sample of successfully uploaded authors:\n")
    for i, result in enumerate(results['success'][:5]):
        print(f"{i+1}. {result['author_name']}")
        print(f"   Author ID: {result['author_id']}")
        print(f"   Abstracts processed: {result['num_abstracts']}")
        print(f"   Embedding dim: {result['response'].get('embedding_dim', 'N/A')}")
        print(f"   Message: {result['response'].get('message', 'N/A')}")
        print()

## 10. Test Search Functionality

In [ ]:
# Test text-based search
def search_authors(query_text: str, limit: int = 5) -> dict:
    """
    Search for authors using text query.
    
    Args:
        query_text: Text query to search for
        limit: Maximum number of results
        
    Returns:
        Search results dictionary
    """
    search_endpoint = f"{VECTOR_DB_BASE_URL}/search/text"
    payload = {
        "query_text": query_text,
        "limit": limit
    }
    
    response = requests.post(search_endpoint, json=payload, timeout=30)
    response.raise_for_status()
    return response.json()

# Example search queries
test_queries = [
    "machine learning and artificial intelligence",
    "network security and cybersecurity",
    "quantum computing research"
]

print("Testing search functionality with sample queries:\n")
for query in test_queries:
    print(f"Query: '{query}'")
    print("-" * 60)
    
    try:
        results = search_authors(query, limit=3)
        
        print(f"Search time: {results.get('search_time_ms', 0):.2f}ms")
        print(f"Found {len(results.get('results', []))} results:\n")
        
        for i, result in enumerate(results.get('results', []), 1):
            entity = result.get('entity', {})
            print(f"  {i}. {entity.get('author_name', 'Unknown')}")
            print(f"     Distance: {result.get('distance', 0):.4f}")
            print(f"     Abstracts: {entity.get('num_abstracts', 0)}")
            print()
    
    except requests.exceptions.RequestException as e:
        print(f"  ✗ Search failed: {e}\n")
    
    print()

## Summary

This notebook successfully:
- ✓ Connected to Azure Blob Storage
- ✓ Loaded author and works data
- ✓ Grouped abstracts by author
- ✓ Uploaded author embeddings to the vector database
- ✓ Tested search functionality

The vector database now contains author embeddings based on their paper abstracts, enabling semantic search for researchers based on their work content.